In [2]:
import pandas as pd
import nltk
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\Smart
[nltk_data]     Computers\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
df = pd.read_csv("software_requirements_extended.csv")
df.head()


,Type,Requirement
0,PE,The system shall refresh the display every 60 ...
1,LF,The application shall match the color of the s...
2,US,If projected the data must be readable. On ...
3,A,The product shall be available during normal ...
4,US,If projected the data must be understandable...


In [4]:
df.columns


Index(['Type', 'Requirement'], dtype='object')

In [5]:
df = df[['Requirement', 'Type']]
df.dropna(inplace=True)

df.head()


,Requirement,Type
0,The system shall refresh the display every 60 ...,PE
1,The application shall match the color of the s...,LF
2,If projected the data must be readable. On ...,US
3,The product shall be available during normal ...,A
4,If projected the data must be understandable...,US


In [6]:
from sklearn.model_selection import train_test_split

X = df['Requirement']
y = df['Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 781
Testing samples: 196


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training vector shape:", X_train_vec.shape)
print("Testing vector shape:", X_test_vec.shape)


Training vector shape: (781, 1884)
Testing vector shape: (196, 1884)


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

print("Model training completed")


Model training completed


In [9]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.6275510204081632

Classification Report:

              precision    recall  f1-score   support

           A       0.00      0.00      0.00         5
           F       0.51      0.95      0.66        41
          FR       0.68      0.97      0.80        65
          FT       0.00      0.00      0.00         4
           L       0.00      0.00      0.00         3
          LF       0.00      0.00      0.00         8
          MN       0.00      0.00      0.00         5
         NFR       0.56      0.25      0.34        20
           O       0.80      0.25      0.38        16
          PE       1.00      0.60      0.75         5
          SC       0.00      0.00      0.00         6
          SE       1.00      0.44      0.62         9
          US       0.83      0.56      0.67         9

    accuracy                           0.63       196
   macro avg       0.41      0.31      0.32       196
weighted avg       0.56      0.63      0.55       196



C:\Users\Smart Computers\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Smart Computers\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Smart Computers\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [10]:
# Convert multiple classes into 2 main categories

def simplify_type(x):
    if x in ['FR', 'F', 'FT', 'O', 'US']:
        return "Functional"
    else:
        return "Non-Functional"

df['Simplified_Type'] = df['Type'].apply(simplify_type)

print(df['Simplified_Type'].value_counts())


Simplified_Type
Functional        652
Non-Functional    325
Name: count, dtype: int64


In [11]:
# New X and y
X = df['Requirement']
y = df['Simplified_Type']

# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.7551020408163265

Classification Report:

                precision    recall  f1-score   support

    Functional       0.75      0.96      0.84       135
Non-Functional       0.76      0.31      0.44        61

      accuracy                           0.76       196
     macro avg       0.76      0.63      0.64       196
  weighted avg       0.76      0.76      0.72       196



In [12]:
# Save model
joblib.dump(model, "srs_classifier_model.pkl")

# Save vectorizer
joblib.dump(vectorizer, "srs_vectorizer.pkl")

print("Model and vectorizer saved successfully ")

Model and vectorizer saved successfully 


In [13]:
def get_similar_requirements(user_prompt, top_n=5):
    
    # Convert all dataset requirements into vectors
    all_requirements = vectorizer.transform(df['Requirement'])
    
    # Convert user input
    user_vec = vectorizer.transform([user_prompt])
    
    # Calculate similarity
    similarity = cosine_similarity(user_vec, all_requirements)
    
    # Get top matches
    top_indices = similarity.argsort()[0][-top_n:]
    
    # Return matched requirements
    return df.iloc[top_indices][['Requirement', 'Simplified_Type']]
    

# Test
results = get_similar_requirements("online shopping system")

print(results)

                                           Requirement Simplified_Type
571  From the here the user can either proceed to c...      Functional
566  From the here the user can either proceed to c...      Functional
38   Any disputes cases that have been closed for o...      Functional
58   The product shall achieve a 98% uptime. The pr...  Non-Functional
469  100% of saved user preferences shall be restor...      Functional


In [19]:
def generate_srs(user_prompt):
    
    results = get_similar_requirements(user_prompt, top_n=30)
    
    include_keywords = ["user", "order", "cart", "payment", "product", "checkout", "customer", "purchase"]
    exclude_keywords = ["chat", "textbox", "api", "phone", "message", "ai", "extract"]
    
    def is_functional_relevant(req):
        req_lower = req.lower()
        if not any(word in req_lower for word in include_keywords):
            return False
        if any(word in req_lower for word in exclude_keywords):
            return False
        return True
    
    def clean_text(text):
        return text.strip().lower().replace(".", "")
    
    # Split
    functional_df = results[results['Simplified_Type'] == 'Functional'].copy()
    non_functional_df = results[results['Simplified_Type'] == 'Non-Functional'].copy()
    
    # Filter only functional
    functional_df = functional_df[functional_df['Requirement'].apply(is_functional_relevant)]
    
    # Clean + deduplicate
    functional_df['clean_req'] = functional_df['Requirement'].apply(clean_text)
    non_functional_df['clean_req'] = non_functional_df['Requirement'].apply(clean_text)
    
    functional_df = functional_df.drop_duplicates(subset="clean_req")
    non_functional_df = non_functional_df.drop_duplicates(subset="clean_req")
    
    functional = list(functional_df['Requirement'].head(5))
    non_functional = list(non_functional_df['Requirement'].head(5))
    
    srs = f"""
    SOFTWARE REQUIREMENT SPECIFICATION

    1. Introduction
    This document describes the requirements for: {user_prompt}

    2. Functional Requirements
    """
    
    if len(functional) == 0:
        srs += "\n    No relevant functional requirements found in dataset."
    else:
        for i, req in enumerate(functional, 1):
            srs += f"\n    {i}. {req}"
    
    srs += "\n\n    3. Non-Functional Requirements\n"
    
    if len(non_functional) == 0:
        srs += "\n    No relevant non-functional requirements found in dataset."
    else:
        for i, req in enumerate(non_functional, 1):
            srs += f"\n    {i}. {req}"
    
    srs += "\n\n    4. Conclusion\n    This SRS is generated purely from dataset using similarity-based retrieval."
    
    return srs


print(generate_srs("online shopping system"))


    SOFTWARE REQUIREMENT SPECIFICATION

    1. Introduction
    This document describes the requirements for: online shopping system

    2. Functional Requirements
    
    1. From the here the user can either proceed to call a rep for an order or proceed to the online checkout process
    2. 100% of saved user preferences shall be restored when system comes back online.

    3. Non-Functional Requirements

    1. The product shall adhere to the corporate online availability schedule.  The application is brought down only within 98% of the scheduled outages per the availability schedule.
    2. The product shall achieve a 98% uptime. The product shall not fail more than 2% of the available online time.

    4. Conclusion
    This SRS is generated purely from dataset using similarity-based retrieval.
